In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

In [ ]:
# Load embeddings and inputs
embeddings = np.load("../embeddings/esm3_embeddings_9912.npz", allow_pickle=True)
gfp_inputs = pd.read_csv("../embeddings/avgfp_plm_inputs.csv")

In [ ]:
import os
# Load split indices (taken from METL)
splits = {}
for file in os.listdir("splits"):
    with open(os.path.join("splits", file), 'r') as file:
        lines = file.readlines()
        split_name = file.name.split("/")[1].split(".")[0]
        splits[split_name] = [int(line[:-1]) for line in lines] # remove new line characters

In [63]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

In [64]:
class CustomDataset(Dataset):
    def __init__(self, dataX, dataY):
        self.X = dataX
        self.y = dataY

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [94]:
# Baseline ESM-3 embeddings
X = np.array(embeddings[embeddings.files[1]])
X_train = X[splits["train"]]
X_test = X[splits["test"]]
X_val = X[splits["val"]]

# ESM-3 embeddings + fa_sol
X_gfp = np.column_stack((X, np.array(gfp_inputs['fa_sol']).astype(np.float32)))
X_gfp_train = X_gfp[splits["train"]]
X_gfp_test = X_gfp[splits["test"]]
X_gfp_val = X_gfp[splits["val"]]

# Test prediction of Rosetta attraction energy
y = np.array(pd.read_csv("../data/rosetta/attraction_energies.csv")['fa_atr'])
y_train = y[splits["train"]]
y_test = y[splits["test"]]
y_val = y[splits["val"]]

# DataLoaders
train_loader = DataLoader(CustomDataset(X_train, y_train), batch_size=64, shuffle=True)
train_gfp_loader = DataLoader(CustomDataset(X_gfp_train, y_train), batch_size=64, shuffle=True)

val_loader = DataLoader(CustomDataset(X_val, y_val), batch_size=64, shuffle=True)
val_gfp_loader = DataLoader(CustomDataset(X_gfp_val, y_val), batch_size=64, shuffle=True)

In [95]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [96]:
def train(model, epochs, training_loader, val_loader, loss_fn, optimizer):
    train_losses = []
    val_losses = []
    for epoch in range(epochs):
        train_loss = 0
        val_loss = 0

        # Train
        for i, data in enumerate(training_loader):
            model.train()
            inputs, labels = data
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(training_loader)
        print(f'  Epoch {epoch + 1} train loss: {train_loss}')
        train_losses.append(train_loss)

        # Validation
        with torch.no_grad():
            for i, data in enumerate(val_loader):
                model.eval()
                inputs, labels = data
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = loss_fn(outputs, labels)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        print(f'  Epoch {epoch + 1} val loss: {val_loss}')
        val_losses.append(val_loss)

    return train_losses, val_losses

In [97]:
EPOCHS = 10

model = NeuralNetwork(X.shape[1])
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
train_losses, val_losses = train(model, EPOCHS, train_loader, val_loader, loss_fn, optimizer)

/Users/mairinoshaughnessy/biophysics-plm/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:630: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/Users/mairinoshaughnessy/biophysics-plm/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:630: UserWarning: Using a target size (torch.Size([44])) that is different to the input size (torch.Size([44, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/Users/mairinoshaughnessy/biophysics-plm/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:630: UserWarning: Using a target size (torch.Size([46])) that is different to the input size (torch.Size([46, 1])). This will likely lead

  Epoch 1 train loss: 232680.4935430241
  Epoch 1 val loss: 878.2531558378188
  Epoch 2 train loss: 421.2739074269962
  Epoch 2 val loss: 266.84000113295355
  Epoch 3 train loss: 246.57414229673844
  Epoch 3 val loss: 171.8267120911372
  Epoch 4 train loss: 196.91798784575263
  Epoch 4 val loss: 154.15371914519497
  Epoch 5 train loss: 174.97864877222867
  Epoch 5 val loss: 134.12336825476697
  Epoch 6 train loss: 160.12220564924823
  Epoch 6 val loss: 111.43255329307065
  Epoch 7 train loss: 149.70384882086518
  Epoch 7 val loss: 101.20837878298335
  Epoch 8 train loss: 156.73838390066837
  Epoch 8 val loss: 97.36003297387676
  Epoch 9 train loss: 141.29722582077537
  Epoch 9 val loss: 109.87160832926205
  Epoch 10 train loss: 141.0131309684105
  Epoch 10 val loss: 86.51388378546874


In [98]:
model_gfp = NeuralNetwork(X_gfp.shape[1])
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model_gfp.parameters(), lr=0.01)
train_gfp_losses, val_gfp_losses = train(model_gfp, EPOCHS, train_gfp_loader, val_gfp_loader, loss_fn, optimizer)

  Epoch 1 train loss: 254562.5887705923
  Epoch 1 val loss: 369.8894912064826
  Epoch 2 train loss: 231.20559092099754
  Epoch 2 val loss: 130.79608847514922
  Epoch 3 train loss: 171.63259840824222
  Epoch 3 val loss: 218.37242301038307
  Epoch 4 train loss: 150.4281678712225
  Epoch 4 val loss: 94.85661439416712
  Epoch 5 train loss: 149.99030854053083
  Epoch 5 val loss: 80.74620177352675
  Epoch 6 train loss: 148.37144082392967
  Epoch 6 val loss: 111.48857416255707
  Epoch 7 train loss: 153.39875777719834
  Epoch 7 val loss: 233.79450113029242
  Epoch 8 train loss: 150.6042821820723
  Epoch 8 val loss: 82.14104160341131
  Epoch 9 train loss: 134.121248723517
  Epoch 9 val loss: 131.51576288122246
  Epoch 10 train loss: 137.15443110370865
  Epoch 10 val loss: 73.20352478965609
